In [52]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas(desc='pandas_bar')

In [53]:
train_df = pd.read_csv('../data/train.csv',index_col='Posting_ID')
test_df = pd.read_csv('../data/test_ans.csv', index_col='Posting_ID', usecols=['Posting_ID', 'Posting_Image_Phash', 'Posting_Title'])
train_df

,Posting_Image_Phash,Posting_Title
Posting_ID,,
84753,b9d0c6ce39306ec6,Miyako Pemanggang Roti TSK 258
181991,be4e80b72561c3f2,McDoDo 1.8M Kabel Gaming Fast Charging 3.0 Mic...
208951,b893c9c696c48ecd,[VOLUMIZER] LAMEILA VOLUMIZER SPAN MASCARA / L...
293636,bf60c63f90980fb2,Maspion Panda Fit Set A
320509,e4db93641932979c,tempat botol minum sepeda turanza alloy
...,...,...
4294232503,e798986763656598,Claris Deluxe Gayung Air 0318
4294325542,b333cc32cccccccc,Minyak Telon Plus Konicare
4294536428,9b0f0c2331ccecf9,Sebamed Baby Protective Facial Cream 50ml ORIG...


In [54]:
# 檢查長度並刪除不符合條件的記錄
invalid_records = []

for index, row in train_df.iterrows():
    if len(row['Posting_Image_Phash']) != 16:
        invalid_records.append(row)
        train_df.drop(index, inplace=True)

# 輸出不符合條件的記錄
if invalid_records != []:
    print("不符合條件的記錄:")
    for record in invalid_records:
        print(record)

print(train_df.shape)

不符合條件的記錄:
Posting_Image_Phash                                 9.79E+108
Posting_Title          Loyang kue bolu 18 cm termurah bandung
Name: 1238500437, dtype: object
(34229, 2)


In [55]:
# 檢查長度並刪除不符合條件的記錄
invalid_records = []

for index, row in test_df.iterrows():
    if len(row['Posting_Image_Phash']) != 16:
        invalid_records.append(row)
        test_df.drop(index, inplace=True)

# 輸出不符合條件的記錄
if invalid_records != []:
    print("不符合條件的記錄:")
    for record in invalid_records:
        print(record)

print(test_df.shape)

(20, 2)


In [56]:
int('b',16)

11

In [57]:
bin(int('b9d0c6ce39306ec6',16))

'0b1011100111010000110001101100111000111001001100000110111011000110'

In [58]:
All_df = pd.concat([train_df,test_df], axis = 0)
All_df['to_binary'] = All_df['Posting_Image_Phash'].apply(lambda x:[int(i) for i in (bin(int(x,16))[2:])])
All_df

,Posting_Image_Phash,Posting_Title,to_binary
Posting_ID,,,
84753,b9d0c6ce39306ec6,Miyako Pemanggang Roti TSK 258,"[1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, ..."
181991,be4e80b72561c3f2,McDoDo 1.8M Kabel Gaming Fast Charging 3.0 Mic...,"[1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, ..."
208951,b893c9c696c48ecd,[VOLUMIZER] LAMEILA VOLUMIZER SPAN MASCARA / L...,"[1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, ..."
293636,bf60c63f90980fb2,Maspion Panda Fit Set A,"[1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, ..."
320509,e4db93641932979c,tempat botol minum sepeda turanza alloy,"[1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, ..."
...,...,...,...
3009299991,ab2fd4d02b27d0d8,Willsen Onemed Tes Hamil Test Pack Tes Kehamil...,"[1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, ..."
2792135571,941d61e13e3cb2e3,Headset / Handsfree U19 macaron / Macaroon Mat...,"[1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, ..."
2346278642,d445e5450eba7b46,Panci Elektrik Serbaguna Multifungsi Panci Lis...,"[1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, ..."


In [59]:
type(All_df['to_binary'].values)

numpy.ndarray

In [60]:
shape0 = All_df.shape[0]
All_df_idx = All_df.index
to_binary_list = All_df.iloc[:train_df.shape[0],:]['to_binary'].values.tolist()

In [65]:
def distance(a_:np.ndarray):
    tmp = np.array([a_] * train_df.shape[0])
    every_dis = pd.Series(np.sum(np.abs(tmp-to_binary_list),axis=1),index=train_df.index)
    return every_dis.sort_values()[:100].index.tolist()

to_binary_list = All_df.iloc[:train_df.shape[0],:]['to_binary'].values.tolist()
test_subset = All_df.iloc[train_df.shape[0]:,:].copy()
#test_subset['nearest_ids'] = test_subset['to_binary'].progress_apply(lambda x:distance(x)
test_subset.loc[:, 'nearest_ids'] = test_subset['to_binary'].progress_apply(lambda x: distance(x))
#test_subset.to_pickle('../output_temp/test_subsets_temp.pkl')

pandas_bar: 100%|██████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.71it/s]


In [5]:
# test_subset = pd.read_pickle('../output_temp/test_subsets_temp.pkl')
# test_subset
#shape0 = All_df.shape[0]
#All_df_idx = All_df.index

In [17]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
All_df['embedding'] = All_df['Posting_Title'].progress_apply(lambda x:model.encode(x))

# model.encode(all_df['Posting_Title'].tolist())

pandas_bar: 100%|████████████████████████████████████████████████████████████████| 34249/34249 [12:09<00:00, 46.92it/s]


In [18]:
All_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 34249 entries, 84753 to 1015625393
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Posting_Image_Phash  34249 non-null  object
 1   Posting_Title        34249 non-null  object
 2   to_binary            34249 non-null  object
 3   embedding            34249 non-null  object
dtypes: object(4)
memory usage: 1.3+ MB


In [66]:
All_df_idx = All_df.iloc[:train_df.shape[0],:].index
embeddings = All_df.iloc[:train_df.shape[0],:]['embedding'].tolist()
shape0 = All_df.iloc[:train_df.shape[0],:].shape[0]

def cos_sim(e_):
    cosSim_lst = [util.pytorch_cos_sim(e_, embeddings[idx]) for idx in range(shape0)]
    every_sim = pd.Series(cosSim_lst,index=All_df_idx)
    return every_sim.sort_values(ascending=False)[:100].index.tolist()

test_cos = All_df.iloc[train_df.shape[0]:,:]
#test_subset['cosSim_id'] = test_cos['embedding'].progress_apply(lambda x:cos_sim(x)
test_subset.loc[:, 'cosSim_id'] = test_cos['embedding'].progress_apply(lambda x: cos_sim(x))



# def cos_sim(e_):
#     for i in shape0:
#         print(util.cos_sim(e_, all_df['embedding'].values[i]))
#         raise ValueError
#     every_sim = pd.Series(util.cos_sim(tmp, all_df['Posting_Title']),index=All_df_idx)
#

# all_df['cos_sim'] = all_df['embedding'].progress_apply(lambda x:cos_sim(x))


KeyError: 'embedding'

In [9]:
test_cos

,Posting_Image_Phash,Posting_Title,to_binary,embedding
Posting_ID,,,,
1805210125,b09bcb648e3a3933,[IMPOR] DOMMO - D2009 Kaos Kaki Polos Semata K...,"[1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, ...","[-0.15620145, 0.063033596, 0.027847279, -0.054..."
2856183961,ee9690c347ceb02e,SABUN PEMUTIH TZUKI ORIGINAL PENGHILANG JERAWA...,"[1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, ...","[-0.061827835, 0.15692946, -0.020807814, 0.046..."
2458596748,eadad1350f6c7842,Pushop Waistbag Pria Traveling Waterproof Prof...,"[1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, ...","[-0.020024382, 0.05878297, -0.016579099, 0.032..."
3879247647,fc8e8e9e87713061,(Bundling 2 Pcs) Acnol Lotion 10Ml Acne Lotion...,"[1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, ...","[-0.078624584, 0.095061235, 0.016890636, -0.00..."
3225117984,f2fbc112ae655845,JILBAB SAFETY MASK NIQAB ANTI CORONA / JILBAB ...,"[1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, ...","[-0.1077712, 0.11973496, -0.004493565, -0.0636..."
2712572834,9d7aea4da067051e,DF - Hijab Khimar DeQiara free tuspin // Tudun...,"[1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, ...","[-0.13042632, 0.014816706, -0.04673302, -0.067..."
1824818423,ada4c4781f93686e,Packingan Tisu Tempered Glass Case Casing Anti...,"[1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, ...","[-0.020796891, 0.008860557, 0.036889676, -0.02..."
2354107204,f848e387bd3a5864,Stand Holder Hp Flexible 360 Gurita Car Holder...,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, ...","[-0.04432967, 0.075525075, -0.073347606, -0.07..."
2442928452,f8b6dbc341c126b4,Tutup Pentil LED / Pentil Ban LED ban motor mo...,"[1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, ...","[-0.061786134, 0.06313467, -0.027428465, 0.003..."


In [44]:
ans = test_subset.copy()
ans['ans_my'] = ans.apply(lambda x: list(set(x['nearest_ids'][:25]).union(set(x['cosSim_id'][:25])) - set(x.index)), axis=1)
ans['ans_my']

Posting_ID
1805210125    [4026107649, 4189113094, 702603526, 4277958290...
2856183961    [2628404353, 510780042, 1157207566, 1091539729...
2458596748    [1771711744, 1859864960, 905556102, 641827208,...
3879247647    [1206963201, 2556644237, 2629377165, 163760782...
3225117984    [3887565568, 720083329, 1718767748, 3298015877...
2712572834    [3369714307, 25936391, 2215864841, 1049431436,...
1824818423    [3740184963, 2904440967, 1602104591, 427301595...
2354107204    [2182605696, 165587202, 3005965446, 651310855,...
2442928452    [3852899328, 735936896, 4071389067, 3932894475...
889623137     [1041890432, 876622850, 626759172, 2005728902,...
1858791960    [3884647552, 1433170692, 1099355015, 232105588...
1211751940    [17278464, 3286790274, 1697329283, 1843059976,...
1915993360    [3226556167, 2901373320, 2834448264, 341172596...
2104971635    [3177654145, 1921036809, 1925181961, 365532302...
2517685633    [2486422784, 2132078728, 502424333, 4072650638...
3009299991    [1264798465, 91

In [45]:
ans['ans_my'].to_csv('ans.csv')

In [46]:
test_results = pd.read_csv('../data/test_results.csv',index_col='Posting_ID')
test_results['ans'] = test_results['ans'].apply(lambda x: [int(i) for i in x.split(',')])
test_results['ans']

Posting_ID
1805210125    [1989561782, 1773789522, 1833837149, 262762843...
2856183961    [1734882388, 3043787667, 1420726013, 376606672...
2458596748    [1065906242, 1652717296, 219315523, 4181834509...
3879247647    [3378866976, 674056726, 1457505137, 2465941424...
3225117984    [470412641, 1462484558, 1323768754, 3647070982...
2712572834    [117475510, 530911546, 169657511, 4089718734, ...
1824818423    [925544261, 3855533455, 95201212, 1475710807, ...
2354107204    [1375793402, 2780411704, 2190988910, 137440880...
2442928452    [1146285530, 1873419172, 287217465, 387726735,...
889623137     [1766600575, 1617944572, 1949926993, 408249847...
1858791960    [4183994442, 2711780517, 3227415372, 360082531...
1211751940    [608120401, 503422247, 3568427243, 2850506812,...
1915993360    [2834448264, 331194711, 2234021757, 3031235277...
2104971635    [3573325557, 2128176597, 1491109987, 403191459...
2517685633    [2021356091, 1349865803, 1625711585, 248642278...
3009299991    [2864876417, 20

In [47]:
concat_result = pd.concat([ans['ans_my'],test_results['ans']],axis=1,ignore_index=True)
concat_result.columns=['my','rst']

In [48]:
concat_result['intersection'] = concat_result.apply(lambda x:set(x['my']) & set(x['rst']),axis=1)
concat_result['cnt_myAns'] = concat_result['my'].apply(lambda x:len(x))
concat_result['cnt_rstAns'] = concat_result['rst'].apply(lambda x:len(x))
concat_result['cnt_intersection'] = concat_result['intersection'].apply(lambda x:len(x))
concat_result['recall'] = concat_result['cnt_intersection']/concat_result['cnt_rstAns']
concat_result['precision'] = concat_result['cnt_intersection']/concat_result['cnt_myAns']
concat_result['f1'] = (
        2.0*concat_result['recall']*concat_result['precision']/(concat_result['recall']+concat_result['precision'])
)
concat_result

,my,rst,intersection,cnt_myAns,cnt_rstAns,cnt_intersection,recall,precision,f1
Posting_ID,,,,,,,,,
1805210125,"[4026107649, 4189113094, 702603526, 4277958290...","[1989561782, 1773789522, 1833837149, 262762843...","{2461306403, 4189113094, 702603526, 1765041994...",50,44,12,0.272727,0.240000,0.255319
2856183961,"[2628404353, 510780042, 1157207566, 1091539729...","[1734882388, 3043787667, 1420726013, 376606672...","{2628404353, 510780042, 1085240851, 3137411481...",48,40,19,0.475000,0.395833,0.431818
2458596748,"[1771711744, 1859864960, 905556102, 641827208,...","[1065906242, 1652717296, 219315523, 4181834509...","{1357710802, 2408563652, 1543316237, 2930367783}",48,36,4,0.111111,0.083333,0.095238
3879247647,"[1206963201, 2556644237, 2629377165, 163760782...","[3378866976, 674056726, 1457505137, 2465941424...","{2629377165, 163760782, 3398839446, 1698463128...",49,35,22,0.628571,0.448980,0.523810
3225117984,"[3887565568, 720083329, 1718767748, 3298015877...","[470412641, 1462484558, 1323768754, 3647070982...","{470412641, 143260486, 3647070982, 3402693191,...",50,34,13,0.382353,0.260000,0.309524
2712572834,"[3369714307, 25936391, 2215864841, 1049431436,...","[117475510, 530911546, 169657511, 4089718734, ...","{1778175585, 3369714307, 169657511, 2384763368...",46,34,15,0.441176,0.326087,0.375000
1824818423,"[3740184963, 2904440967, 1602104591, 427301595...","[925544261, 3855533455, 95201212, 1475710807, ...","{3740184963, 925544261, 3394916135, 498188103,...",49,34,15,0.441176,0.306122,0.361446
2354107204,"[2182605696, 165587202, 3005965446, 651310855,...","[1375793402, 2780411704, 2190988910, 137440880...","{2845480992, 1374408802, 2751055011, 103331682...",49,34,18,0.529412,0.367347,0.433735
2442928452,"[3852899328, 735936896, 4071389067, 3932894475...","[1146285530, 1873419172, 287217465, 387726735,...","{387726735, 1345649171, 2920436119, 2834390170...",50,33,23,0.696970,0.460000,0.554217


In [49]:
concat_result['f1'].mean()

0.42003030355812143

In [50]:
concat_result['precision'].mean()

0.3600578135798101

In [51]:
concat_result['recall'].mean()

0.5089428807935686